In [555]:
import os
import sys

notebook_dir = os.getcwd()
target_folder = os.path.abspath(os.path.join(notebook_dir, ".."))
if target_folder not in sys.path:
    sys.path.append(target_folder)

import cards
import random

random.seed(42)

def random_seven(deck:list):
    board = random.sample(deck, 7)
    return board 

RANDOM = random_seven(cards.DECK)
SF   = ('9d','8d','7d','6d','5d','Ah','Kc')
TRAP = ('9s','7s','5s','3s','2s','8h','6d')
WHEEL= ('As','5d','4c','3h','2s','9d','Kc')
BOAT = ('As','Ah','Ad','Ks','Kh','Kd','Qs')
PAIR = ('Jd','Js','Ks','9h','7c','5d','2c')

In [556]:
def convert_ranks(ranks:dict):
    mapped_ranks = {}
    for rank , count in ranks.items():
        value = cards.CARD_VALUE[rank]
        mapped_ranks[value] = count
    
    return mapped_ranks


def count_ranks_suits(seven:tuple):
    ranks = {}
    suits = {}
    for card in seven:
        ranks[card[0]] = ranks.get(card[0], 0) + 1
        suits[card[1]] = suits.get(card[1], 0) + 1
    
    mapped_ranks = convert_ranks(ranks)

    return mapped_ranks , suits

ranks, suits = count_ranks_suits(BOAT)
print(f'Suits: {suits}')
print(f'Ranks: {ranks}')

Suits: {'s': 3, 'h': 2, 'd': 2}
Ranks: {12: 3, 11: 3, 10: 1}


In [557]:
def find_groups(ranks:dict):
    group_counts = {}
    for rank, count in sorted(ranks.items(), reverse=True):
        group_counts.setdefault(count , []).append(rank)
    
    return group_counts

find_groups(ranks)

{3: [12, 11], 1: [10]}

In [558]:
def find_flush_suit(suits:dict):
    flush_suit = None
    for suit , count in suits.items():
        if count >= 5:
            flush_suit = suit
    
    return flush_suit

flush_suit = find_flush_suit(suits)

In [559]:
def find_flush_cards(seven, suit):
    flush_cards = []
    for card in seven:
        if card[1] == suit:
            flush_cards.append(card)
    
    flush_cards_tuple = tuple(flush_cards)
    
    return flush_cards_tuple

flush_cards = find_flush_cards(BOAT, flush_suit)

flush_cards

()

In [560]:
def find_straight(ranks):
    sorted_ranks = sorted(list(set(ranks)))
    straight_high = None
    consecutive_count = 1
    for i in range(len(sorted_ranks) - 1):
        if sorted_ranks[i] == sorted_ranks[i + 1] - 1:
            consecutive_count += 1
            if consecutive_count >= 5:
                straight_high = sorted_ranks[i + 1]
        else:
            consecutive_count = 1
    
    if straight_high is None:
        if {12,0,1,2,3}.issubset(set(ranks)):
            straight_high = 3

    return straight_high 

find_straight(ranks)

In [564]:
def find_straight_flush(flush_cards):
    flush_ranks, flush_suits = count_ranks_suits(flush_cards)
    flush_straight_high = find_straight(flush_ranks)
    return flush_straight_high

find_straight_flush(flush_cards)


In [ ]:
def evaluate_hand(seven):
    ranks, suits = count_ranks_suits(seven)
    groups = find_groups(ranks)
    flush_suit = find_flush_suit(suits)
    flush_cards = None
    sf_high = None
    if flush_suit is not None:
        flush_cards = find_flush_cards(seven, flush_suit)
        sf_high = find_straight_flush(flush_cards)
    
    straight_high = find_straight(ranks)

    if sf_high is not None:
        return (9, sf_high)
    elif groups.get(4) is not None:
        kicker = list[groups.values()][1]
        return (8, groups.get(4), kicker)
    elif groups.get(3) is not None:
        return (7, groups.get(3).value[0], groups.get(3).value[1])
    elif flush_cards is not None:
        return (6, flush_cards[0],flush_cards[1],flush_cards[2],flush_cards[3],flush_cards[4],)
    elif straight_high is not None:
        return (5, straight_high)
    elif groups.get(3) is not None:
        k1 = list[groups.values()][1]
        k2 = list[groups.values()][2]
        return (4, trips, k1, k2)
    elif groups.get(2) is not None:
        k1 = list[groups.values()][1]
        k2 = list[groups.values()][2]
        k3 = list[groups.values()][3]
        return (3, pair, k1, k2, k3)
    else:
        return (1)
    

In [576]:
evaluate_hand(BOAT)

TypeError: list[dict_values([[12, 11], [10]])] is not a generic class